## Escalonamento de horários

In [21]:
%pip install pulp

import pulp
import pandas as pd

# Demanda (d1 a d7) - dias da semana
d = {1: 3, 2: 4, 3: 5, 4: 3, 5: 4, 6: 5, 7: 6}

prob = pulp.LpProblem("Escalonamento", pulp.LpMinimize)

x = {i: pulp.LpVariable(f"x{i}", lowBound=0, cat='Integer') for i in range(1, 8)}

# Restricoes
prob += x[5] + x[6] + x[7] + x[1] >= d[1]   # domingo
prob += x[6] + x[7] + x[1] + x[2] >= d[2]   # segunda
prob += x[7] + x[1] + x[2] + x[3] >= d[3]   # terça
prob += x[1] + x[2] + x[3] + x[4] >= d[4]   # quarta
prob += x[2] + x[3] + x[4] + x[5] >= d[5]   # quinta
prob += x[3] + x[4] + x[5] + x[6] >= d[6]   # sexta
prob += x[4] + x[5] + x[6] + x[7] >= d[7]   # sábado

# Funcao obj: minimizar o total de enfermeiras
prob += pulp.lpSum(x[i] for i in range(1, 8))

prob.solve(pulp.PULP_CBC_CMD(msg=False))

print(f"\nTotal de enfermeiras: {sum(x[i].varValue for i in range(1, 8))}\n")

day_names = {1: 'Domingo', 2: 'Segunda', 3: 'Terça', 4: 'Quarta', 5: 'Quinta', 6: 'Sexta', 7: 'Sábado'}

# Cronograma
schedule_data = []
nurse_id = 1
for start_day in range(1, 8):
  count = int(x[start_day].varValue)
  for _ in range(count):
    work_days = [(start_day + i - 1) % 7 + 1 for i in range(4)]   # 4 dias consecutivos a partir de start_day
    rest_days = [(start_day + 4 + i - 1) % 7 + 1 for i in range(3)] # 3 dias de descanso
    schedule_data.append({
      'Enfermeira': nurse_id,
      'Dia de Início': day_names[start_day],
      'Dias de Trabalho': ', '.join(day_names[d] for d in work_days),
      'Dias de Descanso': ', '.join(day_names[d] for d in rest_days)
    })
    nurse_id += 1

schedule_df = pd.DataFrame(schedule_data)
print("Cronograma das Enfermeiras:")
display(schedule_df)

# Tabela de cobertura diaria
cobertura_data = []
for day in range(1,8):
    working = sum(1 for nurse in schedule_data if day_names[day] in nurse['Dias de Trabalho'])
    cobertura_data.append({
        'Dia': day_names[day],
        'Demanda': d[day],
        'Enfermeiras Alocadas': working,
        'Atendido?': '✔' if working >= d[day] else '✖'
    })
cobertura_df = pd.DataFrame(cobertura_data)
print("\nVerificação de cobertura por dia:")
display(cobertura_df)

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.

Total de enfermeiras: 8.0

Cronograma das Enfermeiras:


,Enfermeira,Dia de Início,Dias de Trabalho,Dias de Descanso
0,1,Terça,"Terça, Quarta, Quinta, Sexta","Sábado, Domingo, Segunda"
1,2,Terça,"Terça, Quarta, Quinta, Sexta","Sábado, Domingo, Segunda"
2,3,Quarta,"Quarta, Quinta, Sexta, Sábado","Domingo, Segunda, Terça"
3,4,Quinta,"Quinta, Sexta, Sábado, Domingo","Segunda, Terça, Quarta"
4,5,Sexta,"Sexta, Sábado, Domingo, Segunda","Terça, Quarta, Quinta"
5,6,Sábado,"Sábado, Domingo, Segunda, Terça","Quarta, Quinta, Sexta"
6,7,Sábado,"Sábado, Domingo, Segunda, Terça","Quarta, Quinta, Sexta"
7,8,Sábado,"Sábado, Domingo, Segunda, Terça","Quarta, Quinta, Sexta"



Verificação de cobertura por dia:


,Dia,Demanda,Enfermeiras Alocadas,Atendido?
0,Domingo,3,5,✔
1,Segunda,4,4,✔
2,Terça,5,5,✔
3,Quarta,3,3,✔
4,Quinta,4,4,✔
5,Sexta,5,5,✔
6,Sábado,6,6,✔
